## TEXT ANALYTICS
### DATASET EXPLORATION

The names of the genres are (case sensitive): Metal, rock, rap, pop, country.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [9]:
train = Path("..")  / "Dataset" / "initial_train.csv"
test = Path("..")  / "Dataset" / "initial_test.csv"

In [10]:
data = pd.read_csv(train, dtype=str)
data2 = pd.read_csv(test, dtype=str)

In [8]:
data

,Unnamed: 0,Lyric,genre
0,0,"See me, ancient one! Dismal Tuat, Nergal unsaf...",Metal
1,1,Feels like Im covered in lies so turn off the ...,Metal
2,2,"Works of art, painted black Magniloquent, blee...",Metal
3,3,Into the cage like an animal You must survive ...,Metal
4,4,Paralysed in pleasure I hear you call Lost my ...,Metal
...,...,...,...
499995,499995,[Verse 1] I dont want to tell you that its ove...,country
499996,499996,I get to thinking sometimes I dont know why I ...,country
499997,499997,When I was A little boy around the table athom...,country
499998,499998,[Verse 1] Its a junked out joint off a backroa...,country


In [18]:
data2

,Unnamed: 0,Lyric,genre
0,0,can you hear me call your name Im not far away...,Metal
1,1,You say you are so clever You beleive that you...,Metal
2,2,Walking across these misery plains When all fo...,Metal
3,3,Fuck you you bitch get out of my head Twisting...,Metal
4,4,Crashing forth upon the soil Filthy waves gave...,Metal
...,...,...,...
49995,49995,[Verse 1] When the sun sinks down and dreams s...,country
49996,49996,I watched from the window as she slipped from ...,country
49997,49997,"Look around, it is never far See who the wound...",country
49998,49998,We started arguing on the onramp Of Interstate...,country


In [11]:
data[data['genre'] == 'pop']

,Unnamed: 0,Lyric,genre
300000,300000,money good friends mistake amend easy push com...,pop
300001,300001,You left him to die in his bed You left with h...,pop
300002,300002,"Danger, see it in your life Always be aware th...",pop
300003,300003,So many times i stood around And never spoke m...,pop
300004,300004,Well my friends I feel its time To reassess th...,pop
...,...,...,...
399995,399995,I stand in the distance I view from a far Shou...,pop
399996,399996,[Verse 1] Tell me all about your neighbor Tell...,pop
399997,399997,"[Verse 2] You keep talkin, who you trying to k...",pop
399998,399998,[Verse 1] Shes a hotel senator From a nightsta...,pop


There are some mistakes:

 I've seen a song with French lyrics and another in half Hungarian half English,

 Several with text like "Jus Oborn Guitars, Vocals Liz Buckingham Guitars Mark Greening Drums, Percussion" or "Ixithra Vocals, All instruments, Music, Lyrics" instead of lyrics.

 Several also have [Verse Brian, Howie AJ], or [IntroChorus] in the middle of the lyrics
 


 These will need to be fixed.

 For the first maybe apply a language recognizer?

 For the second we could filter by lenght.
 
 For the third filter out the [] ?

I think we can keep the train/test split as it is, maybe it already takes authors into account. 

In [43]:
df_data=pd.concat([data, data2]).reset_index(drop=True)
df_data

,Unnamed: 0,Lyric,genre
0,0,"See me, ancient one! Dismal Tuat, Nergal unsaf...",Metal
1,1,Feels like Im covered in lies so turn off the ...,Metal
2,2,"Works of art, painted black Magniloquent, blee...",Metal
3,3,Into the cage like an animal You must survive ...,Metal
4,4,Paralysed in pleasure I hear you call Lost my ...,Metal
...,...,...,...
549995,49995,[Verse 1] When the sun sinks down and dreams s...,country
549996,49996,I watched from the window as she slipped from ...,country
549997,49997,"Look around, it is never far See who the wound...",country
549998,49998,We started arguing on the onramp Of Interstate...,country


### Data Analysis

In [12]:
print("Number of columns in the dataset:", len(data.columns))

Number of columns in the dataset: 3


In [13]:
# COLUMN TYPES
for col in data.columns:
    print(f"Column '{col}' has type {data[col].dtype}.")

Column 'Unnamed: 0' has type object.
Column 'Lyric' has type object.
Column 'genre' has type object.


In [14]:
print("Number of records in the dataset:", data.shape[0])

Number of records in the dataset: 500000


In [15]:
# MISSING VALUES
for col in data.columns:
    missing_count = data[col].isnull().sum()
    print(f"Column '{col}' has {missing_count} missing values (Percentage: {missing_count/len(data)*100:.2f}%).")

Column 'Unnamed: 0' has 0 missing values (Percentage: 0.00%).
Column 'Lyric' has 0 missing values (Percentage: 0.00%).
Column 'genre' has 0 missing values (Percentage: 0.00%).


In [16]:
# DISTINCT VALUES
for col in data.columns:
    distinct_values = data[col].nunique()
    print(f"Column '{col}' has {distinct_values} distinct values (Percentage: {distinct_values/len(data)*100:.2f}%).")

Column 'Unnamed: 0' has 500000 distinct values (Percentage: 100.00%).
Column 'Lyric' has 496343 distinct values (Percentage: 99.27%).
Column 'genre' has 5 distinct values (Percentage: 0.00%).


In [17]:
# Rows with all duplicate values
duplicate_rows = data[data.duplicated()]
print(f"Number of duplicate rows in the dataset: {len(duplicate_rows)}")

Number of duplicate rows in the dataset: 0


Obvious due to the index.

In [14]:
# Rows with duplicate "Lyric" column values:
duplicate_lyrics = data[data.duplicated(subset="Lyric", keep=False)]
print(f"Number of duplicate lyrics in the dataset: {len(duplicate_lyrics)}")

Number of duplicate lyrics in the dataset: 7272


In [15]:
# Show the duplicate rows
duplicate_lyrics

,Unnamed: 0,Lyric,genre
0,0,"See me, ancient one! Dismal Tuat, Nergal unsaf...",Metal
2,2,"Works of art, painted black Magniloquent, blee...",Metal
39,39,Time never waited He is not your best friend W...,Metal
120,120,Baptized in pitys robes Ive failed to dam the ...,Metal
140,140,Wake up with a scream the fear is written in y...,Metal
...,...,...,...
498304,498304,Those memories of you still haunt me Every nig...,country
498809,498809,Its true you have me twisted around your finge...,country
498896,498896,Its funny how some people dream They put on a ...,country
499429,499429,"[Verse 1 Johnny Cash] The crops are all in, an...",country


!!! Remove them!?

### Data Preparation

No imputing of missing values needed.

Makes no sense to have duplicate rows (same information), so we drop them.

In [44]:
df_data_full=df_data.copy()

#### drop columns

In [45]:
df_data_full = df_data_full.drop(columns=['Unnamed: 0'])
df_data_full

,Lyric,genre
0,"See me, ancient one! Dismal Tuat, Nergal unsaf...",Metal
1,Feels like Im covered in lies so turn off the ...,Metal
2,"Works of art, painted black Magniloquent, blee...",Metal
3,Into the cage like an animal You must survive ...,Metal
4,Paralysed in pleasure I hear you call Lost my ...,Metal
...,...,...
549995,[Verse 1] When the sun sinks down and dreams s...,country
549996,I watched from the window as she slipped from ...,country
549997,"Look around, it is never far See who the wound...",country
549998,We started arguing on the onramp Of Interstate...,country


#### drop duplicates

In [46]:
df_data_full[df_data_full.duplicated(keep=False)].sort_values(by='Lyric')

,Lyric,genre
419860,19 cryin When you said goodbye They say that y...,country
543893,19 cryin When you said goodbye They say that y...,country
48483,"A bad place, a dark space The lines that weath...",Metal
88492,"A bad place, a dark space The lines that weath...",Metal
98712,A frantic journey in your soul Protect your mi...,Metal
...,...,...
70756,step right up march push crawl right up on you...,Metal
264199,verse 1isaiah mott Whose the man of this thing...,rap
267223,verse 1isaiah mott Whose the man of this thing...,rap
29654,withered rope you hang whats empty cant remain...,Metal


In [47]:
df_data_full = df_data_full.drop_duplicates()
len(df_data_full)

549315

In [48]:
duplicates=df_data_full[df_data_full['Lyric'].duplicated(keep=False)].sort_values(by='Lyric').copy()
duplicates

,Lyric,genre
359856,... And the night got swollen Bloated like a m...,pop
503905,... And the night got swollen Bloated like a m...,Metal
325277,".....Everybody listen to me You better stop, d...",pop
33598,".....Everybody listen to me You better stop, d...",Metal
396138,...Flustering in smell of nighttime forest I c...,pop
...,...,...
161274,[Verse 1] You never wear a stitch of lace Your...,rock
369944,[Verse] You dont know what love is Until youve...,pop
525585,[Verse] You dont know what love is Until youve...,rap
20396,breathing the cold air haunting like nightmare...,Metal


In [49]:
duplicates['genre'].value_counts()

genre
Metal      3366
pop        2176
rock       1878
country     147
rap          30
Name: count, dtype: int64

In [51]:
df_data_full=df_data_full[~df_data_full['Lyric'].duplicated(keep=False)]
df_data_full

,Lyric,genre
1,Feels like Im covered in lies so turn off the ...,Metal
2,"Works of art, painted black Magniloquent, blee...",Metal
3,Into the cage like an animal You must survive ...,Metal
4,Paralysed in pleasure I hear you call Lost my ...,Metal
5,"Unseen, cheap, faithful weapon Buried devils a...",Metal
...,...,...
549995,[Verse 1] When the sun sinks down and dreams s...,country
549996,I watched from the window as she slipped from ...,country
549997,"Look around, it is never far See who the wound...",country
549998,We started arguing on the onramp Of Interstate...,country


In [52]:
df_data_full['genre'].value_counts()

genre
rap        109931
country    109691
rock       108102
pop        107626
Metal      106368
Name: count, dtype: int64

#### TEXT PROCESSING

In [53]:
import re

In [54]:
df_data_full["words"] = df_data_full["Lyric"].str.split().str.len()
df_data_full

,Lyric,genre,words
1,Feels like Im covered in lies so turn off the ...,Metal,179
2,"Works of art, painted black Magniloquent, blee...",Metal,121
3,Into the cage like an animal You must survive ...,Metal,140
4,Paralysed in pleasure I hear you call Lost my ...,Metal,178
5,"Unseen, cheap, faithful weapon Buried devils a...",Metal,28
...,...,...,...
549995,[Verse 1] When the sun sinks down and dreams s...,country,249
549996,I watched from the window as she slipped from ...,country,203
549997,"Look around, it is never far See who the wound...",country,195
549998,We started arguing on the onramp Of Interstate...,country,283


In [55]:
print(df_data_full["words"].describe().round(1).to_string())

count    541718.0
mean        246.7
std         183.2
min           1.0
25%         138.0
50%         206.0
75%         306.0
max       16511.0


In [57]:
for _, row in df_data_full.sort_values("words").iloc[[0, 1, 2, -1]].iterrows():
    print(f"{row.words:4d} words | {row.genre}: {row.Lyric[:200]}")

   1 words | Metal: Youreokiwillbei
   1 words | rap: Httpssoundcloud.comjlawless1murderinaminutethrowbackbobftchrisbrownfreestyle?injlawless1setsfreshfreestyles
   1 words | rap: Httprap.genius.com520004EminemwhenimgoneJustknowthatimlookingdownonyousmilingandididntfeelathingsobabydontfeelnopainjustsmileback
16511 words | rap: CHAPTER FOUR A FRESH LIGHT UPON ROBERT DE SAINTLOUP Oh, it is unheardof, said my mother. Listen, at my age, one has ceased to be astonished at anything, but I assure you that there could be nothing mo


In [60]:
df_data_full[['Lyric']][df_data_full['Lyric'].str.contains(r"Htt+", na=False)]

,Lyric
32889,Wir wandern Wortlos wie Gestirne. Zu den Monde...
64416,"In emotion, Carried fragements of hate, Astray..."
90492,"Narfa g hitti er nttin var liin, Niflheimahlii..."
105426,Poor Boy One day in my life When i forgot abou...
105635,Am I wrong in saying that with reference to Tr...
...,...
529855,Top 50 fav. rappers by kinglelove 4150 Top 50 ...
530660,[Intro] [Hook Aaron Compton] Wait for me Wait ...
544598,This value can be increased and enhanced with ...
546267,Kkot Studios knows that a large collection of ...


In [63]:
df_data_full[df_data_full['Lyric'].str.contains(r'(?i)^Htt', na=False)].sort_values(by='words')

,Lyric,genre,words
257002,Httpssoundcloud.comjlawless1murderinaminutethr...,rap,1
283325,Httprap.genius.com520004EminemwhenimgoneJustkn...,rap,1
234234,Httpswww.youtube.comchannelUC8Y3fi1TgAOWBg7LrO...,rap,15
278251,Httpwww.youtube.comwatch?v08yToS9Hwqs trying t...,rap,17
275562,Httpssoundcloud.comjoejungwhiteamerikkka Could...,rap,26
437480,Httpsroycompro.comaboutus Cisco Network Router...,country,30
132018,Httpsbrandonshaffermusic.bandcamp.com 1. Aural...,rock,36
202243,Httpswww.youtube.comwatch?vly2rRk9W2VY Still g...,rap,44
245987,Httpwww.mediafire.com...LorisBastrgoldornothin...,rap,48
204746,Httpswww.youtube.comwatch?v3mw8DRVDM24 [Intro]...,rap,89


In [66]:
df_data_full.iloc[313144,0]

'Forced into squalor,condemned to be poor Promised reprieve like a temper a lure Prospect of success is the bait they employ To keep us complacent and stifle our cries SUMISSIONS THE TOOL USED AGAINST US Malleable minds watch the news on TV Fed select information thats deemed safe to see A platable reality prepared behind doors Safe for the masses who care for more SUBMISSIONS TE TOOL USED AGAINST US Consumed by the dollar the greed it breeds Consumed by consumption and purposeless need The watch from their towers the rat race unfold Moderate the millions who do what theyre told SUBMISSIONS THE TOOL USED AGAINST US'